# Optimal resource states for Phase Estimation by A. Luis and J. Peřina

Resource states proposed by A. Luis and J. Peřina (1996) for optimal phase measurements
Adapted from Qualtran v0.7.0. Copyright 2023 Google LLC, Apache-2.0. Frozen commit `bcb17bb25360ed68ce3bc084645db3e929799389`.


In [ ]:
from collections import Counter
from collections.abc import Callable
from typing import cast

import numpy as np
import sympy
import torch
from IPython import display as notebook_display

from flagquantum.algorithms.primitives.luis_perina_state import (
    LuisPerinaInterim,
    LuisPerinaState,
)
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.signature_graph import export_signature_graph_dot
from flagquantum.dynamic import DynamicCircuit
from flagquantum.experimental.dynamic import run_dynamic

SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


def show_signature(name: str, signature: Signature) -> None:
    if all(type(register.bitsize) is int for register in signature):
        dot = export_signature_graph_dot(signature, name)
    else:
        # Symbolic declaration labels retain width, encoding and output lifetime.
        rows = {}
        for register in signature:
            rows[register.name + " width"] = str(register.bitsize)
            rows[register.name + " side"] = str(register.side.name)
            rows[register.name + " encoding"] = str(register.dtype.encoding)
            rows.update({key: str(value) for key, value in register.dtype.parameters})
        dot = export_call_graph_dot(name, {}, annotations={name: rows})
    display(SVG(render_call_graph_svg(dot)))


## `LuisPerinaInterim`
Helper Bloq to prepare an intermediate resource state which can be used in AA

Specifically, this prepares the state

$$
    \sqrt{\frac{1}{2^{m}}}\sum_{n=0}^{2^m - 1}\left(\cos{\left(\frac{\pi(n+1)}{2^m+1}\right)}
    |n\rangle|0\rangle + i\sin{\left(\frac{\pi(n+1)}{2^m+1}\right)}|n\rangle|1\rangle\right)
$$

This is the state obtained after applying the Hadamard on the flag qubit as described in
Eq 19 of https://arxiv.org/pdf/1805.03662.pdf, which can then be used in a single round of
Amplitude Amplification to boost the amplitude of desired resource state to 1.

In [ ]:
# LuisPerinaInterim composes the accepted shared CRz and H emitters.


### Example Instances

In [ ]:
lprs_interim_prep = LuisPerinaInterim(5)


#### Graphical Signature

In [ ]:
show_signature("lprs_interim_prep", lprs_interim_prep.signature)


### Call Graph

In [ ]:
lprs_interim_graph = lprs_interim_prep.source_graph()
# The source notebook limits the graph to one level.
lprs_interim_direct = {
    edge: count
    for edge, count in lprs_interim_graph.direct_calls.items()
    if edge[0] == lprs_interim_graph.root
}
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(lprs_interim_graph.root, lprs_interim_direct)
        )
    )
)
display({child: count for (_, child), count in lprs_interim_direct.items()})


## `LuisPerinaState`
Prepares optimal resource state $\chi_{m}$ proposed by A. Luis and J. Peřina (1996)

Uses a single round of amplitude amplification, as described in Ref 2, to prepare the
resource state from Ref 1 described as

$$
\chi_{m} = \sqrt{\frac{2}{2^m + 1}}\sum_{n=0}^{2^m - 1}\sin{\frac{\pi(n+1)}{2^m+1}}|n\rangle
$$

#### Parameters
 - `bitsize`: The size of the phase register to prepare the resource state on.  

#### References
 - [Optimum phase-shift estimation and the quantum description of the phase difference](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.54.4564). 
 - [Encoding Electronic Spectra in Quantum Circuits with Linear T Complexity](https://arxiv.org/abs/1805.03662). Section II-B


In [ ]:
# LuisPerinaState composes the complete amplitude-amplification call chain.


### Example Instances

In [ ]:
lp_resource_state_small = LuisPerinaState(5)


In [ ]:
lp_resource_state_symbolic = LuisPerinaState(sympy.Symbol("n"))


#### Graphical Signature

In [ ]:
show_signature("lp_resource_state_small", lp_resource_state_small.signature)
show_signature("lp_resource_state_symbolic", lp_resource_state_symbolic.signature)


### Call Graph

In [ ]:
lp_resource_graph = lp_resource_state_small.source_graph()
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(
                lp_resource_graph.root, lp_resource_graph.direct_calls
            )
        )
    )
)
display(lp_resource_graph.leaf_counts)


## Append and separate resource accounts

The original five-bit and symbolic declarations above retain their source signatures and direct calls. This bounded two-bit execution example checks the exact sine window and clean workspace after the measured reflection. Each measurement branch and the exact inverse are covered by the accompanying source-file tests. The symbolic signature is a declaration label diagram. Native mathematical inversion and the original nonreal-reflection adjoint defect are recorded separately in the file audit.


In [ ]:
state = LuisPerinaState(2)
work, slots = state.requirements()
circuit = DynamicCircuit(2 + work, dtype=torch.complex128)
circuit.append(
    state.bind().make_on(
        selection=(0, 1),
        junk=(),
        workspace=tuple(range(2, 2 + work)),
        classical_bits=tuple(range(slots)),
    )
)
execution = run_dynamic(circuit, shots=1, seed=73, strategy="trajectory")
prepared = execution.final_states.detach().numpy().reshape(4, 2**work)
expected = np.sqrt(2 / 5) * np.sin(np.pi * (1 + np.arange(4)) / 5)
np.testing.assert_allclose(prepared[:, 0], expected, atol=1e-12, rtol=0)
np.testing.assert_allclose(prepared[:, 1:], 0, atol=1e-12, rtol=0)
primitive_counts = dict(Counter(g.name for g in circuit.to_ir().instructions))
declared_qec = state.source_gate_cost().asdict()
display(primitive_counts)
display(declared_qec)
